# 02 · Compute Times

對每個里質心，計算到最近的台南市立圖書館的時間 + 距離。

兩個獨立 axis：

- **`METHOD`**：`haversine_30kmh`（直線距離 ÷ 30 km/h，粗略）或 `osrm`（真實道路路徑，慢但準）
- **`PROFILE`**（OSRM only）：`driving`（公開 router.project-osrm.org，車用）或 `walking`（FOSSGIS routed-foot，步行 ~4.5 km/h）

輸出檔依 profile 命名：
- `data/processed/village_to_nearest_library_driving.csv`
- `data/processed/village_to_nearest_library_walking.csv`
- 進度檔：`data/processed/osrm_progress_{PROFILE}.csv`（支援斷點續跑）

OSRM 輸出兩個指標：`time_min`（行車/步行時間，分鐘）+ `distance_km`（路網真實距離）。


In [ ]:
import sys
from pathlib import Path

import geopandas as gpd
import pandas as pd
from tqdm.notebook import tqdm

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from lib.geo import haversine_km, drive_minutes_from_km, safe_centroid_latlon

RAW_DIR = ROOT / "data" / "raw"
PROC_DIR = ROOT / "data" / "processed"
PROC_DIR.mkdir(parents=True, exist_ok=True)

VILLAGES_IN = RAW_DIR / "tainan_villages.geojson"
LIBRARIES_IN = RAW_DIR / "tainan_libraries.csv"

# ===== 設定 =====
METHOD = "osrm"               # "haversine_30kmh" 或 "osrm"
PROFILE = "driving"           # OSRM 用：driving / walking
SPEED_KMH = 30.0              # haversine fallback 用

# 輸出檔依 profile 命名（haversine 視為 driving）
_profile_key = PROFILE if METHOD == "osrm" else "driving"
OUTPUT = PROC_DIR / f"village_to_nearest_library_{_profile_key}.csv"
print(f"METHOD={METHOD}, PROFILE={_profile_key}, OUTPUT={OUTPUT.name}")

In [ ]:
villages = gpd.read_file(VILLAGES_IN)
libraries = pd.read_csv(LIBRARIES_IN)

print(f"Villages: {len(villages)}, Libraries: {len(libraries)}")

# 對每個里算質心，存成新欄位
centroids = villages.geometry.apply(lambda g: safe_centroid_latlon(g, source_crs="EPSG:4326"))
villages["centroid_lat"] = [c[0] for c in centroids]
villages["centroid_lon"] = [c[1] for c in centroids]
villages[["village_id", "village_name", "district", "centroid_lat", "centroid_lon"]].head()

In [ ]:
def nearest_library_haversine(lat: float, lon: float) -> dict:
    distances = libraries.apply(
        lambda r: haversine_km(lat, lon, r["lat"], r["lon"]),
        axis=1,
    )
    idx = distances.idxmin()
    return {
        "nearest_library": libraries.at[idx, "name"],
        "library_lat": libraries.at[idx, "lat"],
        "library_lon": libraries.at[idx, "lon"],
        "distance_km": float(distances.at[idx]),
    }


if METHOD == "haversine_30kmh":
    rows = []
    for _, v in tqdm(villages.iterrows(), total=len(villages), desc="haversine"):
        n = nearest_library_haversine(v["centroid_lat"], v["centroid_lon"])
        rows.append({
            "village_id": v["village_id"],
            "village_name": v["village_name"],
            "district": v["district"],
            "centroid_lat": v["centroid_lat"],
            "centroid_lon": v["centroid_lon"],
            **n,
            "time_min": drive_minutes_from_km(n["distance_km"], speed_kmh=SPEED_KMH),
            "method": METHOD,
        })
    result = pd.DataFrame(rows)
    result.to_csv(OUTPUT, index=False, encoding="utf-8-sig")
    print(f"✅ Saved {len(result)} rows to {OUTPUT}")
    result.head()

In [ ]:
from lib.osrm import OSRMClient, OSRMError, load_progress, save_progress_row

# Profile-specific OSRM endpoints
# - driving: public OSRM demo (車)
# - walking: FOSSGIS server (only one that has the foot profile properly)
OSRM_SERVERS = {
    "driving": ("https://router.project-osrm.org", "driving"),
    "walking": ("https://routing.openstreetmap.de/routed-foot", "walking"),
}

OSRM_PROGRESS = PROC_DIR / f"osrm_progress_{PROFILE}.csv"
N_CANDIDATES = 3
OSRM_REQUEST_DELAY_S = 0.2

# Column order for both progress + final CSV
OSRM_FIELDS = (
    "village_id", "nearest_library", "library_lat", "library_lon",
    "distance_km", "time_min", "method",
)


def compute_osrm_for_village(client, v_lat, v_lon):
    # haversine 排序取 N 個候選
    candidates = libraries.copy()
    candidates["hav_km"] = candidates.apply(
        lambda r: haversine_km(v_lat, v_lon, r["lat"], r["lon"]), axis=1
    )
    top = candidates.nsmallest(N_CANDIDATES, "hav_km")

    best = None
    method = f"osrm_{PROFILE}"
    for _, lib in top.iterrows():
        try:
            time_min, osrm_km = client.route_summary(
                v_lat, v_lon, lib["lat"], lib["lon"]
            )
        except OSRMError:
            continue
        if best is None or time_min < best["time_min"]:
            best = {
                "nearest_library": lib["name"],
                "library_lat": float(lib["lat"]),
                "library_lon": float(lib["lon"]),
                "distance_km": osrm_km,    # OSRM road distance (not haversine)
                "time_min": time_min,
            }

    if best is None:
        # All candidates failed → fallback to haversine straight-line
        fb = nearest_library_haversine(v_lat, v_lon)
        best = {
            **fb,
            "time_min": drive_minutes_from_km(fb["distance_km"], SPEED_KMH),
        }
        method = f"osrm_{PROFILE}_failed_fallback"

    best["method"] = method
    return best


if METHOD == "osrm":
    base_url, prof = OSRM_SERVERS[PROFILE]
    client = OSRMClient(base_url=base_url, profile=prof,
                        request_delay_s=OSRM_REQUEST_DELAY_S)
    print(f"Server: {base_url} (profile={prof})")

    done = load_progress(OSRM_PROGRESS)
    print(f"Resume: already done {len(done)} / {len(villages)} villages")

    todo = villages[~villages["village_id"].isin(done.keys())].copy()
    print(f"To process: {len(todo)}")

    for _, v in tqdm(todo.iterrows(), total=len(todo), desc=f"OSRM/{PROFILE}"):
        r = compute_osrm_for_village(client, v["centroid_lat"], v["centroid_lon"])
        save_progress_row(
            OSRM_PROGRESS,
            fields=OSRM_FIELDS,
            village_id=v["village_id"],
            nearest_library=r["nearest_library"],
            library_lat=r["library_lat"],
            library_lon=r["library_lon"],
            distance_km=r["distance_km"],
            time_min=r["time_min"],
            method=r["method"],
        )

    # Merge progress + village metadata, output final CSV
    done = load_progress(OSRM_PROGRESS)
    rows = []
    for _, v in villages.iterrows():
        d = done.get(v["village_id"])
        if d is None:
            continue
        rows.append({
            "village_id": v["village_id"],
            "village_name": v["village_name"],
            "district": v["district"],
            "centroid_lat": v["centroid_lat"],
            "centroid_lon": v["centroid_lon"],
            **{k: d[k] for k in ["nearest_library", "library_lat", "library_lon", "distance_km", "time_min", "method"]},
        })
    result = pd.DataFrame(rows)
    result.to_csv(OUTPUT, index=False, encoding="utf-8-sig")
    print(f"✅ Saved {len(result)} rows to {OUTPUT}")
    n_fb = result["method"].str.contains("fallback").sum()
    if n_fb:
        print(f"⚠️  {n_fb} villages fell back to haversine due to OSRM errors")

In [ ]:
import numpy as np

result = pd.read_csv(OUTPUT)
print(f"{OUTPUT.name}:")
print(result["time_min"].describe())
print(); print("Distance (km):"); print(result["distance_km"].describe())
print()
print("By time bin:")
bins = [0, 5, 10, 15, 20, 30, 60, 120, np.inf]
print(pd.cut(result["time_min"], bins=bins, right=False).value_counts().sort_index())